In [ ]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 137.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.18
    Uninstalling idna-3.18:
      Successfully uninstalled idna-3.18


Bajar el dataset

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="MM5xwnVNX7kKcVgjnpQG")
project = rf.workspace("segairix").project("cocoa_detection")
version = project.version(1)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Cocoa_Detection-1 in yolo26:: 100%|██████████| 14805/14805 [00:21<00:00, 689.65it/s] 


Eliminar imágenes muy pequeñas

In [ ]:
import os
from PIL import Image

dataset_path = "/content/Cocoa_Detection-1"
min_resolution = 600

# Recorremos las carpetas típicas de YOLO (train, valid, test)
for split in ['train', 'valid', 'test']:
    images_dir = os.path.join(dataset_path, split, 'images')
    labels_dir = os.path.join(dataset_path, split, 'labels')

    if not os.path.exists(images_dir):
        continue

    deleted_count = 0
    for filename in os.listdir(images_dir):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(images_dir, filename)

            try:
                with Image.open(img_path) as img:
                    width, height = img.size

                if width < min_resolution or height < min_resolution:
                    # Eliminar imagen
                    os.remove(img_path)

                    # Intentar eliminar el txt correspondiente
                    label_filename = os.path.splitext(filename)[0] + '.txt'
                    label_path = os.path.join(labels_dir, label_filename)
                    if os.path.exists(label_path):
                        os.remove(label_path)

                    deleted_count += 1
            except Exception as e:
                print(f"Error procesando {filename}: {e}")

    print(f"En '{split}': Se eliminaron {deleted_count} imágenes por baja resolución.")

En 'train': Se eliminaron 2061 imágenes por baja resolución.
En 'valid': Se eliminaron 623 imágenes por baja resolución.
En 'test': Se eliminaron 300 imágenes por baja resolución.


Resize a 1024 con barras negras

In [ ]:
import os
import cv2
import numpy as np

def letterbox_resize(image, labels, target_size=(1024, 1024)):
    ih, iw = image.shape[:2]
    tw, th = target_size
    scale = min(tw / iw, th / ih)
    nw, nh = int(iw * scale), int(ih * scale)

    # Resize maintain aspect ratio
    image_resized = cv2.resize(image, (nw, nh))

    # Create black canvas
    new_image = np.zeros((th, tw, 3), dtype=np.uint8)

    # Offset to center the image
    dx, dy = (tw - nw) // 2, (th - nh) // 2
    new_image[dy:dy+nh, dx:dx+nw] = image_resized

    # Adjust labels (YOLO format: class x_center y_center width height - all normalized)
    new_labels = []
    for label in labels:
        cls, x, y, w, h = label
        # Convert normalized to pixel in resized image, then to pixel in padded image
        new_x = (x * nw + dx) / tw
        new_y = (y * nh + dy) / th
        new_w = (w * nw) / tw
        new_h = (h * nh) / th
        new_labels.append(f"{int(cls)} {new_x:.6f} {new_y:.6f} {new_w:.6f} {new_h:.6f}")

    return new_image, new_labels

dataset_path = "/content/Cocoa_Detection-1"

for split in ['train', 'valid', 'test']:
    images_dir = os.path.join(dataset_path, split, 'images')
    labels_dir = os.path.join(dataset_path, split, 'labels')

    if not os.path.exists(images_dir):
        continue

    print(f"Procesando split: {split}...")
    for filename in os.listdir(images_dir):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(images_dir, filename)
            lbl_path = os.path.join(labels_dir, os.path.splitext(filename)[0] + ".txt")

            # Load image
            image = cv2.imread(img_path)
            if image is None: continue

            # Load labels
            labels = []
            if os.path.exists(lbl_path):
                with open(lbl_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            labels.append([float(x) for x in parts])

            # Process
            new_img, new_lbls = letterbox_resize(image, labels)

            # Save back
            cv2.imwrite(img_path, new_img)
            with open(lbl_path, 'w') as f:
                f.write("\n".join(new_lbls))

print("¡Dataset escalado a 1024x1024 exitosamente!")

Procesando split: train...
Procesando split: valid...
Procesando split: test...
¡Dataset escalado a 1024x1024 exitosamente!


Quitar boundin boxes muy pequeños

In [ ]:
import os

dataset_path = "/content/Cocoa_Detection-1"
# Dimensiones reales de tus imágenes
img_width = 1024.0
img_height = 1024.0

# Umbral basado en área: Queremos conservar objetos que ocupen al menos el equivalente a
min_area_pixels = 15 * 15

for split in ['train', 'valid', 'test']:
    labels_dir = os.path.join(dataset_path, split, 'labels')
    if not os.path.exists(labels_dir):
        continue

    removed_boxes_count = 0
    total_files_processed = 0

    for filename in os.listdir(labels_dir):
        if filename.endswith(".txt"):
            lbl_path = os.path.join(labels_dir, filename)
            valid_labels = []

            with open(lbl_path, 'r') as f:
                lines = f.readlines()

            for line in lines:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, x, y, w, h = map(float, parts)

                    # Convertir el ancho y alto normalizado de YOLO a píxeles reales
                    box_width_px = w * img_width
                    box_height_px = h * img_height

                    # Calcular el área de la caja
                    box_area_px = box_width_px * box_height_px

                    # Verificar si el área cumple el mínimo de 400 píxeles cuadrados
                    if box_area_px >= min_area_pixels:
                        valid_labels.append(line.strip())
                    else:
                        removed_boxes_count += 1

            # Guardar el archivo actualizado
            with open(lbl_path, 'w') as f:
                if valid_labels:
                    # Unimos con saltos de línea y agregamos uno al final por convención YOLO
                    f.write("\n".join(valid_labels) + "\n")
                else:
                    # Si todas las cajas eran demasiado pequeñas y se eliminaron,
                    # vaciamos el archivo. YOLO lo tomará como una "imagen de fondo" (Background Image).
                    f.write("")

            total_files_processed += 1

    print(f"Split {split}: Se eliminaron {removed_boxes_count} cajas demasiado pequeñas en {total_files_processed} archivos.")

print("Limpieza de bounding boxes basada en área completada.")

Split train: Se eliminaron 1123 cajas demasiado pequeñas en 3114 archivos.
Split valid: Se eliminaron 254 cajas demasiado pequeñas en 862 archivos.
Split test: Se eliminaron 209 cajas demasiado pequeñas en 440 archivos.
Limpieza de bounding boxes basada en área completada.


Guardar en un rar

In [ ]:
import os

# Install rar utility
!apt-get install rar

# Define paths
dataset_folder = "Cocoa_Detection-1"
output_rar = "Cocoa_Detection_Processed.rar"

# Create the rar archive
# -r: recursive, -m5: maximum compression
print(f"Comprimiendo {dataset_folder} en {output_rar}...")
!rar a -r -m5 {output_rar} {dataset_folder}

if os.path.exists(output_rar):
    print(f"\n¡Éxito! El archivo {output_rar} ha sido creado.")
    print("Puedes descargarlo haciendo clic derecho sobre él en el menú de 'Archivos' a la izquierda.")
else:
    print("Hubo un error al crear el archivo rar.")

Streaming output truncated to the last 5000 lines.
Adding    Cocoa_Detection-1/train/labels/trainhoriz_1183_jpg.rf.ce8102d6a83d5e746be4b9c734cea4bd.txt      29%  OK 
Adding    Cocoa_Detection-1/train/labels/trainhoriz_1752_jpg.rf.13859a1b96b1b34426c789ec57aaf8b1.txt      29%  OK 
Adding    Cocoa_Detection-1/train/labels/IMM_433_jpg.rf.78a9b9e8940829ea5bb6146fc9dfa670.txt      29%  OK 
Adding    Cocoa_Detection-1/train/labels/IMM_72_jpg.rf.507179e452e346fad81ec9f9537905eb.txt      29%  OK 
Adding    Cocoa_Detection-1/train/labels/IMM_495_jpg.rf.6bd73db65909e72117afd3b623f4802f.txt      29%  OK 
Adding    Cocoa_Detection-1/train/labels/trainhoriz_1582_jpg.rf.92309db3d3c33e6333e5b6efed93dba2.txt      29%  OK 
Adding    Cocoa_Detection-1/train/labels/trainhoriz_94_jpg.rf.2d2d87755690c25f2ded9a1a49eb9876.txt      29%  OK 
Adding    Cocoa_Detection-1/train/labels/trainhoriz_1685_jpg.rf.2ea9dfb40d26d6f4ea57da823c4b9045.txt     